> **Research provenance.** This notebook records the original empirical workflow. Licensed inputs, saved forecasts and private research infrastructure are not distributed. Outputs and attachments are removed; see [notebook configuration](README.md). The maintained public HMM includes post-study correctness hardening and has not been rerun across the complete 2001-2024 historical sample. Saved paper-era HMM results are not replication targets for the maintained implementation.


# Ensemble method runner with annual hyperparameter tuning

This notebook runs the full 21-method ensemble suite using annual randomized hyperparameter tuning where appropriate.

## Design choices
- 2016 stays in the final portfolio prediction window.
- 2016 is **not** hyperparameter tuned.
- Annual calibration-style tuning starts in 2017.
- Only the non-TC portfolio variants are run:
  - `ew`
  - `vw`
  - `mw_full`
  - `mw_spread_shrink`
  - `mw_h_plus_lowneg`

## Method inventory
- Fixed blend: 50/50, 80/20, 20/80
- Online expert weighting: 7 DynamicWeight variants + ReliabilityHedge
- Probabilistic mixture: EM plain + EM tail
- Sequential decision: ContextualBandit baseline/enriched + RLPolicy baseline/enriched
- Conditional gating: MoE row-level baseline/enriched + MoE date-level baseline/enriched

## Important search-space rule
Model-identity switches are fixed per cell:
- DynamicWeight objective is fixed within each of the 7 variants.
- EM plain vs EM tail are separate models; `use_tail` is not toggled by tuning.
- Baseline vs enriched variants preserve their context/feature setup.
- Row-level MoE and date-level MoE remain separate families.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Optional
import json
import os
import re
import sys

import pandas as pd

def configured_directory(variable):
    value = os.environ.get(variable)
    if not value:
        raise RuntimeError(
            f"Set {variable} to the existing research/artifact directory; "
            "see notebooks/README.md for the external dependencies."
        )
    path = Path(value).expanduser().resolve()
    if not path.is_dir():
        raise FileNotFoundError(f"{variable} is not an existing directory.")
    return path

ROOT = configured_directory("FUSION_RESEARCH_ROOT")

# Explicit prediction artifacts: do not rely on auto-discovery.
RF_ROOT = configured_directory("FUSION_RF_ROOT")
RF_PREDS_FP = RF_ROOT / "rf_preds_rolling_streaming" / "rf_preds_rolling_streaming.parquet"

CNN_ROOT = configured_directory("FUSION_CNN_ROOT")
CNN_PREDS_FP = CNN_ROOT / "ensem_res"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(str(ROOT))

from Scripts.Data import dgp_config as dcf
from multimodal_fusion.pipeline.run_ensemble_portfolio import (
    RunEnsemblePortfolioConfig,
    run_ensemble_portfolio,
)
from multimodal_fusion.pipeline.ensemble_pipeline import EnsemblePipelineConfig, EnsembleSignalPipeline
from multimodal_fusion.experimental.manager import EnsembleManager, EnsembleManagerConfig
from multimodal_fusion.experimental.moe_gating import MoEGating
from multimodal_fusion.experimental.moe_gating_date import MoEGatingDate
from multimodal_fusion.experimental.contextual_bandit import ContextualBandit
from multimodal_fusion.experimental.rl_policy import RLPolicy

OUT_BASE = ROOT / "WORK_SPACE" / "ensemble_family_all_methods_annual_tuned_correct"
OUT_BASE.mkdir(parents=True, exist_ok=True)

dcf.CACHE_DIR = (ROOT / "CACHE_DIR").resolve()
dcf.PORTFOLIO = dcf.CACHE_DIR / "PORTFOLIO"
dcf.CACHE_DIR.mkdir(parents=True, exist_ok=True)

WEEKLY_RET_FP = dcf.CACHE_DIR / "us_week_ret.pq"
MOE_FEAT_FP = dcf.CACHE_DIR / "weekly_with_ratios_moe_features_slim.pq"
MOE_KEYS = ["Date", "StockID"]

RUN_ARTIFACTS: Dict[str, Dict[str, Any]] = {}
signals: Dict[str, pd.DataFrame] = {}

print("ROOT:", ROOT)
print("RF_ROOT exists:", RF_ROOT.exists(), RF_ROOT)
print("RF_PREDS_FP exists:", RF_PREDS_FP.exists(), RF_PREDS_FP)
print("CNN_ROOT exists:", CNN_ROOT.exists(), CNN_ROOT)
print("CNN_PREDS_FP exists:", CNN_PREDS_FP.exists(), CNN_PREDS_FP)
print("OUT_BASE:", OUT_BASE)
print("CACHE_DIR:", dcf.CACHE_DIR)
print("WEEKLY_RET_FP exists:", WEEKLY_RET_FP.exists(), WEEKLY_RET_FP)
print("MOE_FEAT_FP exists:", MOE_FEAT_FP.exists(), MOE_FEAT_FP)

if WEEKLY_RET_FP.exists():
    _tmp = pd.read_parquet(WEEKLY_RET_FP, columns=["Date"])
    _tmp["Date"] = pd.to_datetime(_tmp["Date"], errors="coerce")
    print("Weekly return date range:", _tmp["Date"].min(), "->", _tmp["Date"].max())
    del _tmp


## Shared run controls


In [ ]:
START_YEAR = 2016
END_YEAR = 2024

# 2016 is included in predictions but not tuned.
ANNUAL_FIRST_TUNED_YEAR = 2017

FREQ = "week"
COUNTRY = "USA"

CUT = 10
DELAY = 0
DELAY_LIST = [0]

PORTFOLIO_WEIGHT_TYPES = [
    "ew",
    "vw",
    "mw_full",
    "mw_spread_shrink",
    "mw_h_plus_lowneg",
]

TRADABILITY_SCREENS = False
INCLUDE_PRICE_ADV = False
WARM_PERIOD_RET_CACHE = False

ANNUAL_SCORE_WEIGHT_TYPE = "ew"
ANNUAL_SCORE_CUT = 10
ANNUAL_SCORE_DELAY = 0
ANNUAL_RANDOM_SEED = 1729

TUNED_YEARS = list(range(ANNUAL_FIRST_TUNED_YEAR, END_YEAR + 1))

print("Prediction years:", list(range(START_YEAR, END_YEAR + 1)))
print("Tuned years:", TUNED_YEARS)
print("Number of tuned years:", len(TUNED_YEARS))
print("Portfolio weight types:", PORTFOLIO_WEIGHT_TYPES)


## Search budgets


In [ ]:
SEARCH_BUDGETS = {
    # Fixed blends are untuned baselines.
    "reliability_hedge": 6,

    # DynamicWeight: 7 separate objective variants.
    "dynamic_weight_logloss": 6,
    "dynamic_weight_tail_quantile_logloss": 6,
    "dynamic_weight_tail_margin_weighted_logloss": 6,
    "dynamic_weight_focal_logloss": 6,
    "dynamic_weight_rank_auc": 6,
    "dynamic_weight_rank_spearman_ic": 6,
    "dynamic_weight_rank_decile_spread": 6,

    # EM: plain and tail are separate models by design.
    "em_responsibility_plain": 6,
    "em_responsibility_tail": 6,

    # Sequential decision.
    "contextual_bandit_baseline": 6,
    "contextual_bandit_enriched": 6,
    "rl_policy_baseline": 4,
    "rl_policy_enriched": 4,

    # Conditional gating.
    "moe_gating_baseline": 4,
    "moe_gating_enriched": 4,
    "moe_gating_date_baseline": 6,
    "moe_gating_date_enriched": 6,
}

budget_summary = pd.DataFrame(
    [
        {
            "run_name": name,
            "candidates_per_tuned_year": budget,
            "tuned_years": len(TUNED_YEARS),
            "total_candidate_evals": budget * len(TUNED_YEARS),
        }
        for name, budget in SEARCH_BUDGETS.items()
    ]
).sort_values("run_name").reset_index(drop=True)

budget_summary


## Shared enriched feature set


In [ ]:
feature_cols = [
    "days_since_fund_update",
    "log_mcap",
    "abs_log_ret",
    "abs_ret_5d",
    "abs_ret_20d",
    "abs_ret_5d_minus_20d",
    "fresh_fund_7d",
    "fresh_fund_30d",
]

feature_cols


## Shared overlay patch for enriched models

These helpers attach `weekly_with_ratios_moe_features_slim.pq` on demand whenever a model requests
`feature_cols` or `context_cols` but the incoming dataframe does not already contain them.

This patch is applied to:
- `MoEGating`
- `MoEGatingDate`
- `ContextualBandit`
- `RLPolicy`


In [ ]:
if not MOE_FEAT_FP.exists():
    raise FileNotFoundError(f"Missing slim feature file: {MOE_FEAT_FP}")


def _normalize_overlay_keys(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "Date" in out.columns:
        out["Date"] = pd.to_datetime(out["Date"], errors="coerce").dt.normalize()
    if "StockID" in out.columns:
        out["StockID"] = pd.to_numeric(out["StockID"], errors="coerce").astype("Int64")
    return out


def _extract_requested_cols(model) -> list:
    for attr in ["feature_cols", "context_cols"]:
        cols = getattr(model, attr, None)
        if cols:
            return list(cols)

    for cfg_attr in ["config", "cfg"]:
        cfg = getattr(model, cfg_attr, None)
        if isinstance(cfg, dict):
            for key in ["context_cols", "feature_cols"]:
                cols = cfg.get(key, None)
                if cols:
                    return list(cols)
        elif cfg is not None:
            for key in ["context_cols", "feature_cols"]:
                cols = getattr(cfg, key, None)
                if cols:
                    return list(cols)

    for value in getattr(model, "__dict__", {}).values():
        if isinstance(value, dict):
            for key in ["context_cols", "feature_cols"]:
                cols = value.get(key, None)
                if cols:
                    return list(cols)
        else:
            for key in ["context_cols", "feature_cols"]:
                cols = getattr(value, key, None)
                if cols:
                    return list(cols)

    return []


def _attach_side_features_if_needed(df: pd.DataFrame, requested_cols: list) -> pd.DataFrame:
    needed = [c for c in requested_cols if c not in df.columns]
    if not needed:
        return df

    feat = pd.read_parquet(MOE_FEAT_FP, columns=MOE_KEYS + needed)
    feat = _normalize_overlay_keys(feat).drop_duplicates(subset=MOE_KEYS)

    base = _normalize_overlay_keys(df)
    out = base.merge(feat, on=MOE_KEYS, how="left")

    still_missing = [c for c in needed if c not in out.columns]
    if still_missing:
        raise KeyError(f"Still missing after slim feature merge: {still_missing}")

    return out


def _patch_model_class(cls, method_names):
    for method_name in method_names:
        if not hasattr(cls, method_name):
            continue

        orig_name = f"_overlay_orig_{method_name}"
        if not hasattr(cls, orig_name):
            setattr(cls, orig_name, getattr(cls, method_name))

        orig = getattr(cls, orig_name)

        def make_wrapper(orig_func, cls_name, meth_name):
            def wrapper(self, *args, **kwargs):
                requested_cols = _extract_requested_cols(self)

                if requested_cols:
                    if args and isinstance(args[0], pd.DataFrame):
                        df = _attach_side_features_if_needed(args[0], requested_cols)
                        args = (df,) + args[1:]
                    elif "df" in kwargs and isinstance(kwargs["df"], pd.DataFrame):
                        kwargs = dict(kwargs)
                        kwargs["df"] = _attach_side_features_if_needed(kwargs["df"], requested_cols)

                return orig_func(self, *args, **kwargs)

            wrapper.__name__ = f"_patched_{cls_name}_{meth_name}"
            return wrapper

        setattr(cls, method_name, make_wrapper(orig, cls.__name__, method_name))


_patch_model_class(MoEGating, ["fit", "predict"])
_patch_model_class(MoEGatingDate, ["fit", "predict", "predict_weights"])
_patch_model_class(ContextualBandit, ["fit", "predict"])
_patch_model_class(RLPolicy, ["fit", "predict", "predict_weights"])

print("MoEGating.fit:", MoEGating.fit.__name__)
print("MoEGatingDate.fit:", MoEGatingDate.fit.__name__)
print("ContextualBandit.fit:", ContextualBandit.fit.__name__)
print("RLPolicy.fit:", RLPolicy.fit.__name__)


## Shared runner helper


In [ ]:
def _safe_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_")


def run_one_method(
    run_name: str,
    *,
    method: str,
    attach_labels: bool,
    annual_random_search: bool,
    annual_search_budget: Optional[int],
    method_kwargs: Optional[Dict[str, Any]] = None,
) -> pd.DataFrame:
    out_dir = OUT_BASE / _safe_name(run_name)
    out_dir.mkdir(parents=True, exist_ok=True)

    cfg = RunEnsemblePortfolioConfig(
        rf_root=str(RF_ROOT),
        cnn_root=str(CNN_ROOT),
        portfolio_dir=str(out_dir),

        rf_pred_path=str(RF_PREDS_FP),
        cnn_pred_path=str(CNN_PREDS_FP),

        method=str(method),
        method_kwargs=dict(method_kwargs or {}),

        freq=FREQ,
        country=COUNTRY,
        start_year=START_YEAR,
        end_year=END_YEAR,
        delay_list=DELAY_LIST,

        attach_labels=bool(attach_labels),
        label_threshold=0.0,
        label_ret_col=None,

        cut=CUT,
        delay=DELAY,
        verbose=True,

        tradability_screens=TRADABILITY_SCREENS,
        include_price_adv=INCLUDE_PRICE_ADV,
        warm_period_ret_cache=WARM_PERIOD_RET_CACHE,

        portfolio_weight_types=PORTFOLIO_WEIGHT_TYPES,

        annual_random_search=bool(annual_random_search),
        annual_first_tuned_year=ANNUAL_FIRST_TUNED_YEAR,
        annual_search_budget=annual_search_budget,
        annual_random_seed=ANNUAL_RANDOM_SEED,
        annual_score_weight_type=ANNUAL_SCORE_WEIGHT_TYPE,
        annual_score_cut=ANNUAL_SCORE_CUT,
        annual_score_delay=ANNUAL_SCORE_DELAY,
    )

    print("=" * 120)
    print("RUN:", run_name)
    print("METHOD:", method)
    print("ANNUAL TUNING:", annual_random_search)
    print("ANNUAL SEARCH BUDGET:", annual_search_budget)
    print("OUTPUT DIR:", out_dir)

    sig = run_ensemble_portfolio(cfg)

    RUN_ARTIFACTS[run_name] = {
        "name": run_name,
        "method": method,
        "cfg": cfg,
        "signal": sig,
        "out_dir": out_dir,
        "annual_random_search": bool(annual_random_search),
        "annual_search_budget": annual_search_budget,
    }
    signals[run_name] = sig
    return sig


## Fixed blend


In [ ]:
sig_fixed_5050 = run_one_method(
    "fixed_blend_5050",
    method="fixed_blend",
    attach_labels=False,
    annual_random_search=False,
    annual_search_budget=None,
    method_kwargs={"w_cnn": 0.5, "w_rf": 0.5},
)

sig_fixed_8020 = run_one_method(
    "fixed_blend_8020",
    method="fixed_blend",
    attach_labels=False,
    annual_random_search=False,
    annual_search_budget=None,
    method_kwargs={"w_cnn": 0.8, "w_rf": 0.2},
)

sig_fixed_2080 = run_one_method(
    "fixed_blend_2080",
    method="fixed_blend",
    attach_labels=False,
    annual_random_search=False,
    annual_search_budget=None,
    method_kwargs={"w_cnn": 0.2, "w_rf": 0.8},
)

sig_fixed_5050.head(), sig_fixed_8020.head(), sig_fixed_2080.head()


## Reliability hedge


In [ ]:
sig_reliability_hedge = run_one_method(
    "reliability_hedge",
    method="reliability_hedge",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["reliability_hedge"],
    method_kwargs={
        "label_lag": 1,
    },
)

sig_reliability_hedge.head()


## Dynamic-weight family


In [ ]:
sig_dynamic_weight_logloss = run_one_method(
    "dynamic_weight_logloss",
    method="dynamic_weight",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["dynamic_weight_logloss"],
    method_kwargs={
        "update_freq": "week",
        "label_lag": 1,
        "objective": "logloss",
    },
)

sig_dynamic_weight_tail_quantile_logloss = run_one_method(
    "dynamic_weight_tail_quantile_logloss",
    method="dynamic_weight",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["dynamic_weight_tail_quantile_logloss"],
    method_kwargs={
        "update_freq": "week",
        "label_lag": 1,
        "objective": "tail_quantile_logloss",
    },
)

sig_dynamic_weight_tail_margin_weighted_logloss = run_one_method(
    "dynamic_weight_tail_margin_weighted_logloss",
    method="dynamic_weight",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["dynamic_weight_tail_margin_weighted_logloss"],
    method_kwargs={
        "update_freq": "week",
        "label_lag": 1,
        "objective": "tail_margin_weighted_logloss",
    },
)

sig_dynamic_weight_focal_logloss = run_one_method(
    "dynamic_weight_focal_logloss",
    method="dynamic_weight",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["dynamic_weight_focal_logloss"],
    method_kwargs={
        "update_freq": "week",
        "label_lag": 1,
        "objective": "focal_logloss",
    },
)

sig_dynamic_weight_rank_auc = run_one_method(
    "dynamic_weight_rank_auc",
    method="dynamic_weight",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["dynamic_weight_rank_auc"],
    method_kwargs={
        "update_freq": "week",
        "label_lag": 1,
        "objective": "rank_auc",
    },
)

sig_dynamic_weight_rank_spearman_ic = run_one_method(
    "dynamic_weight_rank_spearman_ic",
    method="dynamic_weight",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["dynamic_weight_rank_spearman_ic"],
    method_kwargs={
        "update_freq": "week",
        "label_lag": 1,
        "objective": "rank_spearman_ic",
        "ret_col": "fwd_ret",
    },
)

sig_dynamic_weight_rank_decile_spread = run_one_method(
    "dynamic_weight_rank_decile_spread",
    method="dynamic_weight",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["dynamic_weight_rank_decile_spread"],
    method_kwargs={
        "update_freq": "week",
        "label_lag": 1,
        "objective": "rank_decile_spread",
        "ret_col": "fwd_ret",
    },
)

sig_dynamic_weight_logloss.head()


## EM responsibility family


In [ ]:
sig_em_responsibility_plain = run_one_method(
    "em_responsibility_plain",
    method="em_responsibility",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["em_responsibility_plain"],
    method_kwargs={
        "config": {
            "by_date": True,
            "label_lag": 1,
            "use_tail": False,
        }
    },
)

sig_em_responsibility_tail = run_one_method(
    "em_responsibility_tail",
    method="em_responsibility",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["em_responsibility_tail"],
    method_kwargs={
        "config": {
            "by_date": True,
            "label_lag": 1,
            "use_tail": True,
            "tail_min_n_per_date": 200,
        }
    },
)

sig_em_responsibility_plain.head(), sig_em_responsibility_tail.head()


## Contextual bandit family


In [ ]:
sig_contextual_bandit_baseline = run_one_method(
    "contextual_bandit_baseline",
    method="contextual_bandit",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["contextual_bandit_baseline"],
    method_kwargs={
        "config": {
            "weight_grid": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            "label_lag": 1,
            "full_information": True,
            "include_prob_context": True,
            "include_perf_context": True,
            "perf_use_common_tail": True,
            "random_state": 7,
        }
    },
)

sig_contextual_bandit_enriched = run_one_method(
    "contextual_bandit_enriched",
    method="contextual_bandit",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["contextual_bandit_enriched"],
    method_kwargs={
        "config": {
            "weight_grid": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            "label_lag": 1,
            "full_information": True,
            "context_cols": feature_cols,
            "include_prob_context": True,
            "include_perf_context": True,
            "perf_use_common_tail": True,
            "random_state": 7,
        }
    },
)

sig_contextual_bandit_baseline.head(), sig_contextual_bandit_enriched.head()


## MoE gating family (row-level / stock-focused)


In [ ]:
sig_moe_gating_baseline = run_one_method(
    "moe_gating_baseline",
    method="moe_gating",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["moe_gating_baseline"],
    method_kwargs={
        "include_prob_features": True,
        "walk_forward": True,
        "warm_start": True,
        "label_lag": 1,
        "sample_per_date": 2000,
        "standardize": True,
    },
)

sig_moe_gating_enriched = run_one_method(
    "moe_gating_enriched",
    method="moe_gating",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["moe_gating_enriched"],
    method_kwargs={
        "feature_cols": feature_cols,
        "include_prob_features": True,
        "walk_forward": True,
        "warm_start": True,
        "label_lag": 1,
        "sample_per_date": 2000,
        "standardize": True,
    },
)

sig_moe_gating_baseline.head(), sig_moe_gating_enriched.head()


## MoE gating family (date-level / weekly-focused)


In [ ]:
sig_moe_gating_date_baseline = run_one_method(
    "moe_gating_date_baseline",
    method="moe_gating_date",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["moe_gating_date_baseline"],
    method_kwargs={
        "include_prob_context": True,
        "label_lag": 1,
    },
)

sig_moe_gating_date_enriched = run_one_method(
    "moe_gating_date_enriched",
    method="moe_gating_date",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["moe_gating_date_enriched"],
    method_kwargs={
        "feature_cols": feature_cols,
        "include_prob_context": True,
        "label_lag": 1,
    },
)

sig_moe_gating_date_baseline.head(), sig_moe_gating_date_enriched.head()


## RL policy family


In [ ]:
sig_rl_policy_baseline = run_one_method(
    "rl_policy_baseline",
    method="rl_policy",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["rl_policy_baseline"],
    method_kwargs={
        "config": {
            "weight_grid": (0.0, 0.25, 0.5, 0.75, 1.0),
            "label_lag": 1,
            "tail_common": True,
            "tail_min_n_per_date": 200,
            "context_mode": "auto",
            "include_prob_features": True,
            "grad_clip": 5.0,
            "id_col": "StockID",
            "random_state": 7,
        }
    },
)

sig_rl_policy_enriched = run_one_method(
    "rl_policy_enriched",
    method="rl_policy",
    attach_labels=True,
    annual_random_search=True,
    annual_search_budget=SEARCH_BUDGETS["rl_policy_enriched"],
    method_kwargs={
        "config": {
            "weight_grid": (0.0, 0.25, 0.5, 0.75, 1.0),
            "label_lag": 1,
            "tail_common": True,
            "tail_min_n_per_date": 200,
            "context_mode": "columns",
            "context_cols": feature_cols,
            "include_prob_features": True,
            "grad_clip": 5.0,
            "id_col": "StockID",
            "random_state": 7,
        }
    },
)

sig_rl_policy_baseline.head(), sig_rl_policy_enriched.head()


## Results dictionary


In [ ]:
list(signals.keys())


## Annual tuning logs

This section loads the per-run annual tuning logs, if a run used annual random search.
For the calibration-style tuner, the year column may be named `calibration_year`.


In [ ]:
log_rows = []

for run_name, meta in RUN_ARTIFACTS.items():
    out_dir = meta["out_dir"]
    method = str(meta["method"]).lower().strip()
    log_fp = out_dir / f"annual_tuning_{method}.json"

    if meta["annual_random_search"] and log_fp.exists():
        with open(log_fp, "r", encoding="utf-8") as f:
            tuning_log = json.load(f)

        per_year = pd.DataFrame(tuning_log["years"]).copy()
        if "calibration_year" in per_year.columns and "validation_year" not in per_year.columns:
            per_year["validation_year"] = per_year["calibration_year"]

        per_year["run_name"] = run_name
        per_year["method"] = method

        keep_cols = ["run_name", "method", "test_year"]
        if "validation_year" in per_year.columns:
            keep_cols.append("validation_year")
        keep_cols.extend(["mode", "chosen_score", "chosen_kwargs"])

        log_rows.append(per_year[keep_cols])

if log_rows:
    tuning_summary_df = pd.concat(log_rows, axis=0, ignore_index=True)
else:
    tuning_summary_df = pd.DataFrame()

tuning_summary_df


## Output file check


In [ ]:
output_manifest = []

for run_name, meta in RUN_ARTIFACTS.items():
    out_dir = meta["out_dir"]
    files = sorted(p.name for p in out_dir.glob("*"))
    output_manifest.append(
        {
            "run_name": run_name,
            "method": meta["method"],
            "annual_random_search": meta["annual_random_search"],
            "annual_search_budget": meta["annual_search_budget"],
            "output_dir": str(out_dir),
            "files_found": len(files),
            "files": files,
        }
    )

output_manifest_df = pd.DataFrame(output_manifest)
output_manifest_df
